# RQ2 -- Parameter recovery

*How does the choice of solver propagate into the accuracy of parameters (β, γ) recovered via least-squares estimation on synthetic data?*

Narrates `experiments/E05_landscape.py`, `E06_optimizers.py`, and `E07_solver_recovery.py`.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..").resolve() / "scripts"))
from plot_style import apply_style, SOLVER_COLOR, COMPARTMENT_COLOR
apply_style()

RESULTS = Path("..") / "results"

def load(exp_id):
    return json.loads((RESULTS / f"{exp_id}.json").read_text())


## The cost landscape J(β, γ)

Built by grid search (E05); its curvature and condition number set how hard the estimation problem is.

In [ ]:
d5 = load("E05")
sigma_key = list(d5["surfaces"].keys())[0]
surf = d5["surfaces"][sigma_key]
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.pcolormesh(d5["beta_grid"], d5["gamma_grid"], surf["surface"], shading="auto", cmap="viridis")
ax.set_xlabel("β"); ax.set_ylabel("γ"); ax.set_title(f"J(β,γ), σ={sigma_key}, κ(JᵀJ)≈{surf['cond_jtj']:.1f}")
fig.colorbar(im, ax=ax)
plt.show()


## Optimizer shoot-out (E06)

Success rate, mean iterations, and mean function evaluations for five optimizers over many random starts.

In [ ]:
d6 = load("E06")
for name, r in d6["results"].items():
    print(f"{name:28s}  success={r['success_rate']*100:5.1f}%  iters={r['mean_iterations'] or float('nan'):6.2f}  fevals={r['mean_fevals'] or float('nan'):7.1f}")


## The centrepiece: solver-in-the-loop recovery bias (E07)

For each solver, holding everything else fixed, bias in the recovered β at sigma=0 (noise-free) is caused *purely* by that solver's own truncation error. It should shrink at the solver's theoretical order as h -> 0 -- this is the concrete mechanism behind RQ2.

In [ ]:
d7 = load("E07")
solvers = sorted({c["solver"] for c in d7["cells"]})
theta0 = (d7["cells"][0]["beta_true"], d7["cells"][0]["gamma_true"])
fig, ax = plt.subplots(figsize=(6, 4.5))
for solver in solvers:
    cells = [c for c in d7["cells"] if c["solver"] == solver and c["sigma_frac"] == 0.0
             and (c["beta_true"], c["gamma_true"]) == theta0]
    cells.sort(key=lambda c: c["h"])
    hs = [c["h"] for c in cells]
    bias = [abs(c["bias"][0]) for c in cells]
    ax.loglog(hs, bias, "o-", color=SOLVER_COLOR.get(solver, "gray"), label=solver)
ax.set_xlabel("fitting step size h"); ax.set_ylabel("|bias| in β̂ (noise-free)")
ax.legend(); ax.set_title("Solver-induced bias in recovered β")
plt.show()
